# SLIIT IT3051 - Data Mining & Predictive Analytics
## EV Battery Prognostics & Health Management (PHM) Dual-Task System
### Group: Necrons | Phase 1: EDA, Data Cleaning & Preprocessing (Week 1 / Viva 1)

---

### Team Roles & Modular Work Breakdown:
| Person | Topic Owned | Key Deliverables |
| :--- | :--- | :--- |
| **Person A** | **Data Structure, Schema & Missingness Audit** | Schema inspection, variable types, duplicate checks, 67-column missingness breakdown, sensor sanity screening |
| **Person B** | **Distributions, Outliers & Treatment Decisions** | Univariate distributions, RUL bell-curve analysis, IQR/Z-score outlier detection, physical outlier justification |
| **Person C** | **Imbalance, Multicollinearity & Feature Engineering** | Class imbalance (18,616 vs 1,384), correlation heatmaps, bivariate degradation analysis, 4 domain features |
| **Person D** | **Data Leakage Guard, Splits & ColumnTransformer** | Target Isolation, ID exclusion, stratified train/test split, production ColumnTransformer pipeline |


### 0. Environment Setup & Global Imports


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual formatting settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = 'Arial'

print('Libraries loaded successfully!')


#### D.1 Boundary Conditions & Target Isolation Policy


In [ ]:
# Drop Identifier Columns
id_cols = ['vehicle_id', 'battery_serial']
target_rul = 'predicted_remaining_life_cycles'
target_fail = 'battery_failure'

# Predictor columns candidate list
excluded_features = id_cols + [target_rul, target_fail]
feature_names = [col for col in df_feat.columns if col not in excluded_features]

cat_features = df_feat[feature_names].select_dtypes(include=['object', 'string']).columns.tolist()
num_features = df_feat[feature_names].select_dtypes(include=[np.number]).columns.tolist()

print(f'Excluded Primary IDs: {id_cols}')
print(f'Total Predictors: {len(feature_names)} (Numeric: {len(num_features)}, Categorical: {len(cat_features)})')
print(f'Categorical Columns ({len(cat_features)}): {cat_features}')


#### D.2 Constructing the Production ColumnTransformer Pipeline


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Pipeline for Numerical Features: Median Imputation + Scaling
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline for Categorical Features: Mode Imputation + One-Hot Encoding
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Master ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features)
    ],
    verbose_feature_names_out=False
)

print('ColumnTransformer Pipeline assembled successfully!')


#### D.3 Task 1 Partitioning & Preprocessing: RUL Regression


In [ ]:
from sklearn.model_selection import train_test_split

# Filter out unlabeled target rows for Task 1
mask_t1 = df_feat[target_rul].notnull()
df_t1 = df_feat[mask_t1].copy()

X1 = df_t1[feature_names]
y1 = df_t1[target_rul]

print(f'Task 1 Labeled Sample Size: {len(df_t1):,} rows (dropped {len(df_feat)-len(df_t1):,} unobserved targets)')

# 80/20 Train/Test Split
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.20, random_state=42)

# Fit on train ONLY, transform test to prevent leakage
preprocessor_t1 = ColumnTransformer(
    transformers=[('num', num_pipeline, num_features), ('cat', cat_pipeline, cat_features)],
    verbose_feature_names_out=False
)

X1_train_trans = preprocessor_t1.fit_transform(X1_train)
X1_test_trans = preprocessor_t1.transform(X1_test)

print(f'X1_train shape: {X1_train_trans.shape}')
print(f'X1_test shape:  {X1_test_trans.shape}')
print(f'Total Post-Transformation Features: {X1_train_trans.shape[1]}')


#### D.4 Task 2 Partitioning & Preprocessing: Critical Failure Classification


In [ ]:
# Task 2 uses all 20,000 records
X2 = df_feat[feature_names]
y2 = df_feat[target_fail]

# Stratified 80/20 Train/Test Split preserving 93.08% / 6.92% class balance
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=42, stratify=y2
)

preprocessor_t2 = ColumnTransformer(
    transformers=[('num', num_pipeline, num_features), ('cat', cat_pipeline, cat_features)],
    verbose_feature_names_out=False
)

X2_train_trans = preprocessor_t2.fit_transform(X2_train)
X2_test_trans = preprocessor_t2.transform(X2_test)

print(f'X2_train shape: {X2_train_trans.shape} (Failures: {(y2_train==1).sum():,} [{(y2_train==1).mean()*100:.2f}%])')
print(f'X2_test shape:  {X2_test_trans.shape} (Failures: {(y2_test==1).sum():,} [{(y2_test==1).mean()*100:.2f}%])')
print(f'Leakage Guard Verified: Strict fit_transform on train, transform on test.')


**Person D Viva Notes:**
- Enforced strict **Target Isolation**: `battery_failure` is not in $X_1$, and `predicted_remaining_life_cycles` is not in $X_2$.
- Dropped identifier columns (`vehicle_id`, `battery_serial`) to eliminate memorization risk.
- Implemented a unified `ColumnTransformer` (median + standard scaler for numericals; mode + one-hot encoder with `handle_unknown='ignore'` for categoricals).
- **Zero Data Leakage:** Preprocessors were strictly fitted on the training split only and subsequently applied to test splits.
